# 认识

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from astroquery.nist import Nist
import astropy.units as u

# --- 沿用你之前配置好的美化设置 ---
fonts = ['Microsoft YaHei', 'SimHei', 'SimSun', 'sans-serif']
plt.rcParams['font.sans-serif'] = fonts
sns.set_theme(style="whitegrid", font=fonts)

## 物质波长

In [ ]:
table = Nist.query(300 * u.nm, 700 * u.nm, linename="H I")

- `observed` 实验室实际测得波长
- `Ritz` 理论波长
- `Rel.` 相对光强。

In [10]:
type(table)

astropy.table.table.Table

In [ ]:


def get_astropy_lines(element="H I", wav_min=390, wav_max=660):
    """
    通过 NIST 获取特定元素的谱线数据
    element: "H I" (中性氢), "Ca II" (一价钙离子) 等
    """
    # 查询 NIST 数据库
    table = Nist.query(wav_min * u.nm, wav_max * u.nm, linename=element)
    
    # 转换为 Pandas DataFrame 方便处理
    df = table.to_pandas()
    # 筛选相对强度比较高的线（便于在光谱中识别）
    # NIST 的强度(Rel.)有时是字符串，需要清理
    df['Rel.'] = pd.to_numeric(df['Rel.'], errors='coerce').fillna(0)
    df = df[df['Rel.'] > 50].copy() # 只保留强线
    
    return df[['Observed', 'Rel.', 'Term']]

# 获取氢 (H I) 和 钙 (Ca II) 的真实数据
h_lines = get_astropy_lines("H I", 390, 660)
ca_lines = get_astropy_lines("Ca II", 390, 400)

print("从 NIST 抓取的氢线数据：")
print(h_lines)


KeyError: "['Term'] not in index"

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))

# 绘制氢线 (Hydrogen)
ax.vlines(x=h_lines['Observed'], ymin=0, ymax=h_lines['Rel.'], 
          colors='red', label='H I (NIST 真实数据)', lw=2)

# 绘制钙线 (Calcium)
ax.vlines(x=ca_lines['Observed'], ymin=0, ymax=ca_lines['Rel.'], 
          colors='blue', label='Ca II (NIST 真实数据)', lw=2)

# 标注具体的线名
for _, row in h_lines.iterrows():
    ax.text(row['Observed'], row['Rel.'], f"H {row['Observed']:.1f}", rotation=45)

ax.set_xlabel("波长 (nm)")
ax.set_ylabel("相对强度 (Relative Intensity)")
ax.set_title("基于 Astroquery/NIST 获取的真实原子谱线分布", pad=20)
ax.legend()
plt.tight_layout()
plt.show()
